# Run the test suite

Validates the from-scratch `nmt` package on Colab. Run the cells top to bottom.

1. **Locate + install** finds the repo (the folder containing `pyproject.toml`) and installs the package plus the test deps.
2. **Run all** runs the whole suite; or use the per-file / single-file cells below.

`SKIP` on `test_kv_cache` and the MQA/GQA case in `test_attention` is expected (those variants aren't built yet) - not a failure. `PASSED` is good; paste any `FAILED` / `ERROR` output back to Claude.

## 1. Locate the repo + install

In [10]:
!git clone https://github.com/Rishi-Jain-27/hindi-translator.git /content/hindi-translator

fatal: destination path '/content/hindi-translator' already exists and is not an empty directory.


In [11]:
!pip install -e .

Obtaining file:///content/hindi-translator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nmt (pyproject.toml) ... done
  Created wheel for nmt: filename=nmt-0.1.0-0.editable-py3-none-any.whl size=1279 sha256=023d8f846701ad60e1fc84b84b3efab476ac87ab5cc80a0bfa8a04656c25f871
  Stored in directory: /tmp/pip-ephem-wheel-cache-crs35iey/wheels/bd/c2/ef/ae818ff943d77ea8d63ef07aea61a1b82808716362dc169d4c
Successfully built nmt
  Attempting uninstall: nmt
    Found existing installation: nmt 0.1.0
    Uninstalling nmt-0.1.0:
      Successfully uninstalled nmt-0.1.0


In [12]:
!cd /content/hindi-translator && git pull
!cd /content/hindi-translator && pip install -q -e .


remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.83 KiB | 1.92 MiB/s, done.
From https://github.com/Rishi-Jain-27/hindi-translator
   363d314..44da918  main       -> origin/main
Updating 363d314..44da918
Fast-forward
 notebooks/00_run_tests.ipynb | 363 +++++++++++++++++++++++++++++++------------
 tests/test_translate.py      |  27 +++-
 2 files changed, 291 insertions(+), 99 deletions(-)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nmt (pyproject.toml) ... done


In [13]:
import os

# If auto-detect fails, set this to your repo path, e.g. "/content/hindi-translator" or a Drive path.
REPO_DIR = "/content/hindi-translator"

d = REPO_DIR or os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
assert os.path.exists(os.path.join(d, "pyproject.toml")), "repo not found - set REPO_DIR above"
os.chdir(d)
print("repo root:", d)

repo root: /content/hindi-translator


In [14]:
!pip install -q -e . pytest sentencepiece

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nmt (pyproject.toml) ... done


## 2. Run the whole suite

In [15]:
!python -m pytest tests -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/hindi-translator
configfile: pyproject.toml
plugins: anyio-4.13.0, langsmith-0.7.23, typeguard-4.5.1
collected 81 items                                                             

tests/test_attention.py::test_self_attention_shapes PASSED               [  1%]
tests/test_attention.py::test_cross_attention_shapes PASSED              [  2%]
tests/test_attention.py::test_attention_rows_sum_to_one PASSED           [  3%]
tests/test_attention.py::test_masked_positions_get_zero_weight PASSED    [  4%]
tests/test_attention.py::test_no_nan_when_a_full_row_is_masked PASSED    [  6%]
tests/test_attention.py::test_mqa_gqa_head_broadcasting SKIPPED (MQA...) [  7%]
tests/test_beam.py::test_beam1_matches_greedy_single PASSED              [  8%]
tests/test_beam.py::test_beam1_matches_greedy_longer P

## 3. Or run them one at a time

Runs each file separately with a header, so it's obvious which file fails.

In [16]:
%%bash
for f in test_schedule test_loss test_optim test_masks test_attention test_model_forward \
         test_tokenizer test_data_collate test_ema test_checkpoint test_tracking \
         test_profiling test_loop; do
  echo "===== $f ====="
  python -m pytest tests/$f.py -q
done

===== test_schedule =====
.....                                                                    [100%]
5 passed in 0.02s
===== test_loss =====
....                                                                     [100%]
4 passed in 2.06s
===== test_optim =====
....                                                                     [100%]
4 passed in 3.24s
===== test_masks =====

no tests ran in 0.01s
===== test_attention =====
.....s                                                                   [100%]
5 passed, 1 skipped in 2.07s
===== test_model_forward =====
.....                                                                    [100%]
5 passed in 1.97s
===== test_tokenizer =====
...                                                                      [100%]
3 passed in 0.36s
===== test_data_collate =====
...                                                                      [100%]
3 passed in 1.93s
===== test_ema =====
......                                            

## 4. Run a single file

Edit the filename and run.

In [17]:
!python -m pytest tests/test_loop.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/hindi-translator
configfile: pyproject.toml
plugins: anyio-4.13.0, langsmith-0.7.23, typeguard-4.5.1
collected 2 items                                                              

tests/test_loop.py::test_loss_decreases PASSED                           [ 50%]
tests/test_loop.py::test_runs_with_ema_swa_ckpt PASSED                   [100%]

============================== 2 passed in 4.10s ===============================


## Fallback: if `!` / `%%bash` are blocked

Some VS Code + Colab connections don't pass shell commands through. Use the Python API instead:

In [18]:
import pytest
pytest.main(["tests", "-v"])  # or ["tests/test_loop.py", "-v"] for a single file

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/hindi-translator
configfile: pyproject.toml
plugins: anyio-4.13.0, langsmith-0.7.23, typeguard-4.5.1
collecting ... collected 0 items / 18 errors

==================================== ERRORS ====================================
___________________ ERROR collecting tests/test_attention.py ___________________
ImportError while importing test module '/content/hindi-translator/tests/test_attention.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_attention.py:7: in <module>
    from nmt.config import ModelConfig
E   ModuleNotFoundError: No module named 'nmt'

<ExitCode.INTERRUPTED: 2>